# Project 2

##### Authors: Ole Marius Green & Einar Aarn
##### Date: 2.11.26

## 3: Model problems

Required libraries:

In [13]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp

We define our two model problems by their initial values, fields and potentials.

### 3.1: 2D charged particle

In [14]:
def B_2D(x):
    return np.array([0, 0, np.sqrt(x[0]**2+x[1]**2)])

def A_2D(x):
    a_0 = -x[1]*np.sqrt(x[0]**2+x[1]**2)
    a_1 = x[0]*np.sqrt(x[0]**2+x[1]**2)
    return 1/3*np.array([a_0, a_1, 0])

def U_2D(x):
    return (100*np.sqrt(x[0]**2+x[1]**2))**-1

def div_U_2D(x):
    divisor = (x[0]**2+x[1]**2)**(1.5)
    return -0.01*np.array([x[0]/divisor, x[1]/divisor, 0])

x_0_2D = np.array([0.0, 0.1, 0.1])
v_0_2D = np.array([0.09, 0.05, 0.2])

### 3.2: Tokamak magnetic field

In [16]:
def B_T(x):
    x1 = x[0]
    x2 = x[1]
    x3 = x[2]
    eqlid2 = x1**2+x2**2

    b_0 = -(2*x2+x1*x3)/(2*eqlid2)
    b_1 = (2*x1-x2*x3)/(2*eqlid2)
    b_2 = (np.sqrt(eqlid2)-1)/(2*np.sqrt(eqlid2))

    return np.array([b_0, b_1, b_2])

def A_T(x):
    x1 = x[0]
    x2 = x[1]
    x3 = x[2]
    eqlid2 = x1**2+x2**2

    a_0 = 0
    a_1 = 0.5*(x1-np.log(np.sqrt(eqlid2)+x1))
    a_2 = -0.5*(x3*np.atan(x2/x1) + np.log(eqlid2))

    return np.array([a_0, a_1, a_2])

def U_T(x):
    return 0.0

def div_U_T(x):
    return np.array([0.0, 0.0, 0.0])

## 4: Implementation of methods

### 4.1: Boris method

In [ ]:
def Boris_Method(x0, v0, t_stop, h, B, div_U, rhs_func):
    n_steps = np.ceil(t_stop/h)

    t = 0.0
    x = x0
    v = v0

    t_vals = np.zeros((n_steps + 1, 3))
    x_vals = np.zeros((n_steps + 1, 3))
    v_vals = np.zeros((n_steps + 1, 3))

    t_vals[0] = t
    x_vals[0] = x0
    v_vals[0] = v0

    # Getting the first v minus half
    def rhs(t, u):
        x_ = u[:3]
        v_ = u[3:]
        return np.concatenate((v_, np.cross(v_, B(x_)) - div_U(x_)))
    
    sol0 = solve_ivp(rhs, (t, t+h), np.concatenate((x0, v0)), t_eval = t + h/2)
    v_minus_half = sol0.y[3:, 0]

    #Here, t in the project sheet is named k instead (to not mix with time)
    for i in range(1, n_steps + 1):

        k = 0.5*h*B(x)
        s = 2*k/(1 + t*t)
        v_minus = v_minus_half - 0.5*h*div_U(x)
        v_marked = v_minus + np.cross(v_minus, t)
        v_plus = v_minus + np.cross(v_marked, s)
        v_plus_half = v_plus - 0.5*h*div_U(x)

        t += h
        x += h*v_plus_half
        v = 0.5*(v_minus_half + v_plus_half)
        v_minus_half = v_plus_half

        t_vals[i] = t
        x_vals[i] = x
        v_vals[i] = v

    return t_vals, x_vals, v_vals

### 4.2: Classical Runge-Kutta 4 on the extended system

In [ ]:
def RK_4(x0, v0, t_stop, h, B, div_U):
    n_steps = np.ceil(t_stop/h)

    t = 0.0
    x = x0
    v = v0

    t_vals = np.zeros((n_steps + 1, 3))
    x_vals = np.zeros((n_steps + 1, 3))
    v_vals = np.zeros((n_steps + 1, 3))

    t_vals[0] = t
    x_vals[0] = x0
    v_vals[0] = v0

    def f(x, v):
        return np.cross(v, B(x)) - div_U(x)
    
    for i in range(1, n_steps + 1):
        

        k1v = h*f(x, v)
        k1x = h*v
        k2v = h*f(x + 0.5*k1x, v + 0.5*k1v)
        k2x = h*(v + 0.5*k1v)
        k3v = h*f(x + 0.5*k2x, v + 0.5*k2v)
        k3x = h*(v + 0.5*k2v)
        k4v = h*f(x + k3x, v + k3v)
        k4x = h*(v + k3v)

        
        t += h
        x += 1/6 * (k1x + 2*k2x + 2*k3x + k4x)
        v += 1/6 * (k1v + 2*k2v + 2*k3v + k4v)

        t_vals[i] = t
        x_vals[i] = x
        v_vals[i] = v

    return t_vals, x_vals, v_vals


### 4.3: Fourth-order symmetric multistep method

In [ ]:

def multistep(x0, v0, t_stop, h, A):

    #solve_ivp RK45 with rtol=1e-10, atol=1e-12

    #Defining constants
    a1, a2, a3 = -0.7, 0.1, 0.9
    sum_a = a1 + a2 + a3
    sum_prod2 = a1 * a2 + a1 * a3 + a2 * a3
    prod3 = a1 * a2 * a3
        
    beta_0 = (1.0 / 3.0) * (20 * prod3 - 4 * sum_prod2 - 28 * sum_a - 52)
    beta_1 = (1.0 / 3.0) * (2 * prod3 + 14 * sum_prod2 + 26 * sum_a + 38)

    alpha = np.array([1.0, -7/5, 9/25, 22/125, -34/125, 22/125, 9/25, -7/5, 1.0])

delta_y_j = (1.0 / 12.0) * (y[j-2] - 8.0*y[j-1] + 8.0*y[j+1] - y[j+2])

## 5: Experiments and deliverables

### A. Plotting trajectories and error over long time

### B. Order verification and work precision diagrams

### C. Exploratory exercise 